# Training at Scale — Theory Notebook

**Chapter 15, Section 06**

Interactive implementations of the mathematics behind large-scale LLM training:

1. **Adam Optimiser** — update equations, bias correction, AdamW weight decay
2. **Gradient Clipping** — norm computation and clipping with direction preservation
3. **Learning Rate Schedules** — warmup, cosine decay, WSD (trapezoidal)
4. **Parallelism & Memory** — ZeRO sharding, pipeline bubbles, communication costs
5. **MoE Load Balancing** — router simulation, auxiliary loss, expert capacity
6. **MFU Calculation** — hardware FLOPs utilisation for real models
7. **LoRA Parameter Counting** — trainable vs frozen parameter analysis
8. **Numerical Stability** — initialisation variance, gradient statistics, μP scaling
9. **Full Training Loop Simulation** — end-to-end optimisation with all components
10. **Summary**

In [ ]:
# === Setup ===
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.random.seed(42)

def print_vector(name, v, fmt=".6f"):
    entries = ", ".join(f"{x:{fmt}}" for x in v)
    print(f"  {name} = [{entries}]")

def print_table(headers, rows, col_widths=None):
    if col_widths is None:
        col_widths = [max(len(str(h)), max(len(str(r[i])) for r in rows)) + 2 
                      for i, h in enumerate(headers)]
    header_str = "".join(str(h).ljust(w) for h, w in zip(headers, col_widths))
    print(f"  {header_str}")
    print("  " + "-" * sum(col_widths))
    for row in rows:
        row_str = "".join(str(r).ljust(w) for r, w in zip(row, col_widths))
        print(f"  {row_str}")

print("Setup complete ✓")

## §1 Adam Optimiser — Full Implementation

The Adam optimiser maintains per-parameter first and second moment estimates:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

Bias-corrected estimates:
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

Update rule: $\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

**AdamW** decouples weight decay: $\theta_t = \theta_{t-1} - \eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon} - \eta\lambda\theta_{t-1}$

In [ ]:
# === §1: Adam Optimiser ===

class Adam:
    """Full Adam optimiser with optional weight decay (AdamW)."""
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0):
        self.params = params  # list of numpy arrays
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0
        # Initialise moments to zero
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
    
    def step(self, grads):
        """Perform one Adam update. Returns update magnitudes for analysis."""
        self.t += 1
        updates = []
        for i, (p, g) in enumerate(zip(self.params, grads)):
            # First moment (momentum)
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            # Second moment (adaptive LR)
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            # Adam update
            update = self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
            p -= update
            # AdamW weight decay (decoupled)
            if self.weight_decay > 0:
                p -= self.lr * self.weight_decay * p
            updates.append(update)
        return updates

# --- Worked example: step-by-step Adam update ---
print("=== Adam Step-by-Step (t=1) ===\n")
g = np.array([0.5, -1.2, 0.3])
beta1, beta2, eta, eps = 0.9, 0.999, 0.001, 1e-8

# Step 1: First moment
m1 = (1 - beta1) * g
print("Step 1: First moment")
print(f"  m₁ = (1-β₁)g = {1-beta1} × g")
print_vector("m₁", m1)

# Step 2: Second moment
v1 = (1 - beta2) * g**2
print("\nStep 2: Second moment")
print(f"  v₁ = (1-β₂)g² = {1-beta2} × g²")
print_vector("v₁", v1)

# Step 3: Bias correction
m1_hat = m1 / (1 - beta1**1)
v1_hat = v1 / (1 - beta2**1)
print("\nStep 3: Bias correction")
print(f"  m̂₁ = m₁/(1-β₁¹) = m₁/{1-beta1} = g  (perfect correction at t=1)")
print_vector("m̂₁", m1_hat)
print_vector("v̂₁", v1_hat)

# Step 4: Update
delta_theta = eta * m1_hat / (np.sqrt(v1_hat) + eps)
print("\nStep 4: Parameter update")
print(f"  Δθ = η × m̂₁ / (√v̂₁ + ε)")
print_vector("Δθ", delta_theta)
print(f"\n  Note: all updates ≈ {eta} regardless of gradient magnitude!")
print("  → Adam normalises to approximately unit step size per parameter")

# --- Multi-step training on a toy problem ---
print("\n=== Adam Training on f(x,y) = x² + 10y² ===\n")
np.random.seed(42)
theta = np.array([5.0, 3.0])
opt = Adam([theta], lr=0.1, beta1=0.9, beta2=0.999)

print(f"{'Step':<6} {'θ₀':<12} {'θ₁':<12} {'f(θ)':<12} {'‖g‖':<10} {'‖Δθ‖':<10}")
print("-" * 62)
trajectory = [theta.copy()]

for step in range(30):
    loss = theta[0]**2 + 10 * theta[1]**2
    grad = np.array([2*theta[0], 20*theta[1]])
    grad_norm = np.linalg.norm(grad)
    updates = opt.step([grad])
    update_norm = np.linalg.norm(updates[0])
    trajectory.append(theta.copy())
    if step % 3 == 0:
        print(f"{step:<6} {theta[0]:<12.6f} {theta[1]:<12.6f} {loss:<12.6f} {grad_norm:<10.4f} {update_norm:<10.6f}")

# --- AdamW comparison ---
print("\n=== Adam vs AdamW Weight Decay ===\n")
print("Adam + L2: adds λθ to gradient → interacts with v (adaptive scaling)")
print("AdamW:     applies λθ directly → clean weight decay, independent of gradients")
print()

theta_adam = np.array([5.0, 3.0])
theta_adamw = np.array([5.0, 3.0])
opt_adam = Adam([theta_adam], lr=0.1, beta1=0.9, beta2=0.999, weight_decay=0.0)
opt_adamw = Adam([theta_adamw], lr=0.1, beta1=0.9, beta2=0.999, weight_decay=0.01)

print(f"{'Step':<6} {'Adam ‖θ‖':<14} {'AdamW ‖θ‖':<14} {'Diff'}")
print("-" * 46)
for step in range(20):
    g_adam = np.array([2*theta_adam[0], 20*theta_adam[1]]) + 0.01 * theta_adam  # L2 in gradient
    g_adamw = np.array([2*theta_adamw[0], 20*theta_adamw[1]])
    opt_adam.step([g_adam])
    opt_adamw.step([g_adamw])
    if step % 4 == 0:
        n_adam = np.linalg.norm(theta_adam)
        n_adamw = np.linalg.norm(theta_adamw)
        print(f"{step:<6} {n_adam:<14.6f} {n_adamw:<14.6f} {n_adam-n_adamw:+.6f}")

# Visualization
if HAS_MPL:
    traj = np.array(trajectory)
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    # Loss contour
    x = np.linspace(-6, 6, 100)
    y = np.linspace(-4, 4, 100)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + 10*Y**2
    ax.contour(X, Y, Z, levels=20, alpha=0.3, cmap='viridis')
    ax.plot(traj[:, 0], traj[:, 1], 'r-o', markersize=3, linewidth=1, label='Adam trajectory')
    ax.plot(traj[0, 0], traj[0, 1], 'go', markersize=10, label='Start')
    ax.plot(traj[-1, 0], traj[-1, 1], 'r*', markersize=15, label='End')
    ax.set_xlabel('θ₀'); ax.set_ylabel('θ₁')
    ax.set_title('Adam on f(x,y) = x² + 10y²')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('adam_trajectory.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: adam_trajectory.png")

## §2 Gradient Clipping

Unconstrained gradients can explode, causing divergence. Gradient clipping bounds the update:

**Clip-by-norm:**
$$\mathbf{g}_{\text{clip}} = \begin{cases} \mathbf{g} & \text{if } \|\mathbf{g}\|_2 \le \tau \\ \tau \cdot \frac{\mathbf{g}}{\|\mathbf{g}\|_2} & \text{otherwise} \end{cases}$$

Key property: **direction is preserved**, only magnitude is scaled.

Why not clip per-component? Per-component clipping (clamping) distorts the gradient direction, potentially pointing updates toward saddle points instead of minima.

In [ ]:
# === §2: Gradient Clipping ===

def clip_by_norm(g, tau):
    """Clip gradient vector by global norm, preserving direction."""
    norm = np.linalg.norm(g)
    if norm <= tau:
        return g.copy(), norm, False
    return tau * g / norm, norm, True

def clip_by_value(g, tau):
    """Clip gradient per-component (BAD — distorts direction)."""
    return np.clip(g, -tau, tau)

# --- Worked example: g = [3.0, 4.0, 0.0], τ = 1.0 ---
print("=== Gradient Clipping: g = [3, 4, 0], τ = 1.0 ===\n")
g = np.array([3.0, 4.0, 0.0])
tau = 1.0
norm = np.linalg.norm(g)
g_clipped, _, was_clipped = clip_by_norm(g, tau)
g_value_clipped = clip_by_value(g, tau)

print(f"Original gradient:     g = {g}")
print(f"‖g‖₂ = {norm:.2f} > τ = {tau} → clip triggered")
print(f"\nClip-by-norm result:   {g_clipped}")
print(f"  Direction: g/‖g‖ = {g / norm}")
print(f"  ‖g_clip‖ = {np.linalg.norm(g_clipped):.4f} = τ ✓")
print(f"\nClip-by-value result:  {g_value_clipped}")
print(f"  ‖g_val‖ = {np.linalg.norm(g_value_clipped):.4f}")

# Direction check
cos_norm = np.dot(g, g_clipped) / (np.linalg.norm(g) * np.linalg.norm(g_clipped))
cos_val = np.dot(g, g_value_clipped) / (np.linalg.norm(g) * np.linalg.norm(g_value_clipped))
print(f"\nDirection preservation:")
print(f"  cos(g, g_clip_norm)  = {cos_norm:.6f}  (1.0 = perfect)")
print(f"  cos(g, g_clip_value) = {cos_val:.6f}  (< 1.0 → direction distorted!)")

# --- Simulate training with/without clipping ---
print("\n=== Training Stability: Clipping vs No Clipping ===\n")
np.random.seed(42)

def simulated_training(use_clipping, tau=1.0, steps=30):
    theta = np.array([5.0, 3.0])
    losses = []
    for t in range(steps):
        loss = theta[0]**2 + 10*theta[1]**2
        losses.append(loss)
        grad = np.array([2*theta[0], 20*theta[1]])
        # Simulate occasional gradient spike (like loss spike)
        if t in [5, 15, 25]:
            grad *= 50  # explosion!
        if use_clipping:
            grad, _, _ = clip_by_norm(grad, tau)
        theta -= 0.01 * grad
    return losses

losses_noclip = simulated_training(False)
losses_clip = simulated_training(True, tau=5.0)

print(f"{'Step':<6} {'No Clip':<15} {'With Clip':<15}")
print("-" * 36)
for i in range(0, 30, 3):
    nc = losses_noclip[i] if not np.isinf(losses_noclip[i]) else float('inf')
    wc = losses_clip[i]
    print(f"{i:<6} {nc:<15.4f} {wc:<15.4f}")
print(f"\nFinal loss — No clip: {losses_noclip[-1]:.4f}, With clip: {losses_clip[-1]:.4f}")

# --- Visualize gradient clipping effect ---
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Direction preservation
    ax = axes[0]
    gs = [np.array([3, 4]), np.array([-2, 5]), np.array([1, -6]), np.array([-4, -3])]
    tau = 2.0
    for g_orig in gs:
        g_clip, _, _ = clip_by_norm(g_orig, tau)
        ax.arrow(0, 0, g_orig[0], g_orig[1], head_width=0.15, head_length=0.1, 
                 fc='lightcoral', ec='red', alpha=0.5, linewidth=1.5)
        ax.arrow(0, 0, g_clip[0], g_clip[1], head_width=0.15, head_length=0.1,
                 fc='lightblue', ec='blue', linewidth=2)
    circle = plt.Circle((0, 0), tau, fill=False, color='green', linestyle='--', linewidth=2)
    ax.add_patch(circle)
    ax.set_xlim(-7, 5); ax.set_ylim(-7, 6)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(f'Gradient Clipping (τ={tau})')
    ax.legend(['Original (red)', 'Clipped (blue)', f'τ-ball'], fontsize=9)
    
    # Plot 2: Training loss with/without clipping
    ax = axes[1]
    ax.plot(losses_noclip, 'r-', label='No clipping', linewidth=2)
    ax.plot(losses_clip, 'b-', label='With clipping (τ=5)', linewidth=2)
    for t in [5, 15, 25]:
        ax.axvline(t, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Training Stability'); ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3); ax.set_yscale('symlog')
    
    plt.tight_layout()
    plt.savefig('gradient_clipping.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: gradient_clipping.png")

## §3 Learning Rate Schedules

Large models use multi-phase LR schedules. The standard recipe:

**Linear Warmup** (first $W$ steps):
$$\eta_t = \eta_{\max} \cdot \frac{t}{W}$$

**Cosine Decay** (after warmup):
$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{\pi(t - W)}{T - W}\right)\right)$$

**WSD (Warmup-Stable-Decay)** — emerging alternative:
$$\eta_t = \begin{cases} \eta_{\max} \cdot t/W & t \le W \\ \eta_{\max} & W < t \le S \\ \eta_{\max} \cdot \frac{T - t}{T - S} & t > S \end{cases}$$

Key insight: WSD enables **continual training** — train with stable LR, then branch and decay whenever you need a checkpoint.

In [ ]:
# === §3: Learning Rate Schedules ===

def warmup_cosine(t, eta_max=3e-4, eta_min=3e-5, warmup=2000, total=100000):
    """Standard warmup + cosine annealing."""
    if t < warmup:
        return eta_max * t / warmup
    progress = (t - warmup) / (total - warmup)
    return eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * progress))

def warmup_stable_decay(t, eta_max=3e-4, warmup=2000, stable_end=80000, total=100000):
    """WSD (Warmup-Stable-Decay) trapezoidal schedule."""
    if t < warmup:
        return eta_max * t / warmup
    elif t < stable_end:
        return eta_max
    else:
        return eta_max * (total - t) / (total - stable_end)

def one_cycle(t, eta_max=3e-4, eta_min=3e-6, total=100000):
    """1-cycle policy: ramp up then ramp down."""
    mid = total // 2
    if t < mid:
        return eta_min + (eta_max - eta_min) * t / mid
    else:
        return eta_max - (eta_max - eta_min) * (t - mid) / mid

def inverse_sqrt(t, eta_max=3e-4, warmup=2000):
    """Inverse sqrt schedule (Transformer original paper)."""
    if t < warmup:
        return eta_max * t / warmup
    return eta_max * np.sqrt(warmup / t)

# --- Compute and compare schedules ---
total = 100000
steps = np.arange(total)
schedules = {
    'Warmup+Cosine': [warmup_cosine(t) for t in steps],
    'WSD (Trapezoidal)': [warmup_stable_decay(t) for t in steps],
    '1-Cycle': [one_cycle(t) for t in steps],
    'Inverse √t': [inverse_sqrt(t) for t in steps],
}

# --- Worked example: cosine schedule at specific points ---
print("=== Cosine Schedule: η_max=3e-4, η_min=3e-5, W=2000, T=100K ===\n")
test_points = [0, 1000, 2000, 5000, 25000, 50000, 75000, 99000]
print(f"{'Step':<10} {'Phase':<12} {'η':<15} {'η/η_max'}")
print("-" * 50)
for t in test_points:
    lr = warmup_cosine(t)
    phase = "warmup" if t < 2000 else "cosine"
    print(f"{t:<10} {phase:<12} {lr:<15.6e} {lr/3e-4:.4f}")

# --- Compare decay rates ---
print("\n=== Schedule Comparison at Key Points ===\n")
header = f"{'Step':<10}"
for name in schedules:
    header += f"{name:<20}"
print(header)
print("-" * 90)
for t in [0, 1000, 2000, 10000, 50000, 80000, 95000, 99000]:
    row = f"{t:<10}"
    for name in schedules:
        row += f"{schedules[name][t]:<20.6e}"
    print(row)

# --- Visualization ---
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: All schedules
    ax = axes[0]
    colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
    for (name, lrs), color in zip(schedules.items(), colors):
        ax.plot(steps[::100], lrs[::100], label=name, linewidth=2, color=color)
    ax.set_xlabel('Training Step'); ax.set_ylabel('Learning Rate')
    ax.set_title('LR Schedule Comparison (T=100K)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.ticklabel_format(style='sci', axis='y', scilimits=(-4,-4))
    
    # Plot 2: Warmup zoom-in
    ax = axes[1]
    warmup_end = 5000
    for (name, lrs), color in zip(schedules.items(), colors):
        ax.plot(steps[:warmup_end], lrs[:warmup_end], label=name, linewidth=2, color=color)
    ax.axvline(2000, color='gray', linestyle='--', alpha=0.5, label='Warmup end')
    ax.set_xlabel('Training Step'); ax.set_ylabel('Learning Rate')
    ax.set_title('Warmup Phase (first 5K steps)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.ticklabel_format(style='sci', axis='y', scilimits=(-4,-4))
    
    plt.tight_layout()
    plt.savefig('lr_schedules.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: lr_schedules.png")

## §4 Parallelism Strategies & Memory Analysis

Training a 70B model requires distributing work across hundreds of GPUs. The math determines which strategy to use.

**Memory per GPU (mixed precision training):**
$$M_{\text{total}} = \underbrace{2N}_{\text{params (fp16)}} + \underbrace{2N}_{\text{gradients (fp16)}} + \underbrace{12N}_{\text{optimizer (fp32 copy + m + v)}} = 16N \text{ bytes}$$

Plus activations: $M_{\text{act}} \approx 2 \cdot s \cdot b \cdot h \cdot L \cdot (10 + \frac{s \cdot n_h}{h})$ bytes

**ZeRO Stages** shard optimizer/gradient/parameters across $K$ GPUs:
- Stage 1: Optimizer → $2N + 2N + 12N/K$
- Stage 2: + Gradients → $2N + 2N/K + 12N/K$  
- Stage 3: + Parameters → $2N/K + 2N/K + 12N/K = 16N/K$

**Pipeline Parallelism** — bubble fraction:
$$f_{\text{bubble}} = \frac{p - 1}{p - 1 + m}$$
where $p$ = pipeline stages, $m$ = microbatches.

In [ ]:
# === §4: Parallelism & Memory Analysis ===

def memory_breakdown(N_billion, K_gpus=1, zero_stage=0, seq_len=2048, 
                      batch_per_gpu=1, n_layers=None, hidden=None, n_heads=None,
                      checkpoint_activations=False):
    """
    Calculate GPU memory for training.
    N_billion: model size in billions of parameters
    Returns dict with memory components in GB.
    """
    N = N_billion * 1e9  # total parameters
    
    # Model states
    params_bytes = 2 * N  # fp16
    grads_bytes = 2 * N   # fp16
    opt_bytes = 12 * N    # fp32 copy(4N) + m(4N) + v(4N)
    
    # Apply ZeRO sharding
    if zero_stage == 0:
        params_mem = params_bytes / K_gpus  # DP already splits batch, not model
        grads_mem = grads_bytes
        opt_mem = opt_bytes
    elif zero_stage == 1:
        params_mem = params_bytes
        grads_mem = grads_bytes
        opt_mem = opt_bytes / K_gpus
    elif zero_stage == 2:
        params_mem = params_bytes
        grads_mem = grads_bytes / K_gpus
        opt_mem = opt_bytes / K_gpus
    elif zero_stage == 3:
        params_mem = params_bytes / K_gpus
        grads_mem = grads_bytes / K_gpus
        opt_mem = opt_bytes / K_gpus
    else:
        raise ValueError(f"Invalid ZeRO stage: {zero_stage}")
    
    # Activation memory (rough estimate)
    act_mem = 0
    if n_layers and hidden and n_heads:
        s, b, h, L, nh = seq_len, batch_per_gpu, hidden, n_layers, n_heads
        act_per_layer = 2 * s * b * h * (10 + s * nh / h)  # bytes
        if checkpoint_activations:
            act_mem = act_per_layer * np.sqrt(L)  # checkpoint every √L layers
        else:
            act_mem = act_per_layer * L
    
    to_gb = lambda x: x / (1024**3)
    return {
        'Parameters (fp16)': to_gb(params_mem),
        'Gradients (fp16)': to_gb(grads_mem),
        'Optimizer (fp32)': to_gb(opt_mem),
        'Activations': to_gb(act_mem),
        'Total': to_gb(params_mem + grads_mem + opt_mem + act_mem)
    }

# --- ZeRO comparison for 7B model ---
print("=== ZeRO Memory Analysis: 7B Parameter Model, 8 GPUs ===\n")
print(f"{'Component':<22} {'No ZeRO':<12} {'Stage 1':<12} {'Stage 2':<12} {'Stage 3':<12}")
print("-" * 70)

results = {}
for stage in [0, 1, 2, 3]:
    results[stage] = memory_breakdown(7, K_gpus=8, zero_stage=stage)

for component in ['Parameters (fp16)', 'Gradients (fp16)', 'Optimizer (fp32)', 'Total']:
    row = f"{component:<22}"
    for stage in [0, 1, 2, 3]:
        row += f"{results[stage][component]:<12.1f}"
    print(row)

print("\nKey insight: ZeRO-3 reduces per-GPU memory from "
      f"{results[0]['Total']:.1f} GB to {results[3]['Total']:.1f} GB "
      f"({results[0]['Total']/results[3]['Total']:.1f}× reduction)")

# --- Real model memory requirements ---
print("\n=== Memory Requirements for Real Models ===\n")
models = [
    ("LLaMA-7B", 7, 32, 4096, 32),
    ("LLaMA-13B", 13, 40, 5120, 40),
    ("LLaMA-70B", 70, 80, 8192, 64),
    ("GPT-3 175B", 175, 96, 12288, 96),
    ("LLaMA-405B", 405, 126, 16384, 128),
]

print(f"{'Model':<16} {'Params':<10} {'No ZeRO':<12} {'ZeRO-3/8':<12} {'ZeRO-3/64':<12} {'A100s needed'}")
print("-" * 76)
for name, N, L, H, NH in models:
    no_zero = memory_breakdown(N, 1, 0)['Total']
    z3_8 = memory_breakdown(N, 8, 3)['Total']
    z3_64 = memory_breakdown(N, 64, 3)['Total']
    # A100 80GB, need ~70GB usable
    min_gpus = int(np.ceil(N * 16 / (70 * 1024**3 / 1e9)))
    print(f"{name:<16} {N:<10} {no_zero:<12.1f} {z3_8:<12.1f} {z3_64:<12.1f} {min_gpus}")

# --- Pipeline Parallelism: Bubble Analysis ---
print("\n\n=== Pipeline Parallelism: Bubble Fraction ===\n")

def bubble_fraction(p, m):
    """Fraction of time wasted in pipeline bubble."""
    return (p - 1) / (p - 1 + m)

print(f"{'Stages (p)':<12} {'μ-batches':<12} {'Bubble %':<12} {'Efficiency %'}")
print("-" * 50)
configs = [(4, 4), (4, 8), (4, 16), (4, 32), (8, 8), (8, 16), (8, 32), (8, 64)]
for p, m in configs:
    bf = bubble_fraction(p, m)
    print(f"{p:<12} {m:<12} {bf*100:<12.1f} {(1-bf)*100:.1f}")

print("\nRule of thumb: m ≥ 4p for < 20% bubble overhead")

# --- Communication Cost Analysis ---
print("\n\n=== Communication Primitives ===\n")
# Ring all-reduce: 2(K-1)/K * N_bytes
def all_reduce_time(N_bytes, K_gpus, bandwidth_gbps):
    """Time for ring all-reduce in seconds."""
    data_moved = 2 * (K_gpus - 1) / K_gpus * N_bytes
    return data_moved / (bandwidth_gbps * 1e9 / 8)

print("Ring All-Reduce: total data = 2(K-1)/K × N bytes\n")
print(f"{'Model':<16} {'K GPUs':<10} {'Bandwidth':<14} {'Data (GB)':<12} {'Time (ms)'}")
print("-" * 64)
for name, N_B in [("7B", 7), ("70B", 70), ("175B", 175)]:
    N_bytes = N_B * 1e9 * 2  # fp16 gradients
    for K, bw in [(8, 600), (64, 400), (256, 200)]:
        data_gb = 2 * (K-1)/K * N_bytes / 1e9
        time_s = all_reduce_time(N_bytes, K, bw)
        print(f"{name:<16} {K:<10} {bw} Gbps{'':<6} {data_gb:<12.1f} {time_s*1000:<.1f}")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: ZeRO memory savings
    ax = axes[0]
    model_sizes = [1, 3, 7, 13, 30, 70]
    for stage, color, marker in [(0, 'red', 'o'), (1, 'orange', 's'), 
                                  (2, 'blue', '^'), (3, 'green', 'D')]:
        mems = [memory_breakdown(n, 8, stage)['Total'] for n in model_sizes]
        ax.plot(model_sizes, mems, f'{marker}-', color=color, 
                label=f'ZeRO-{stage}', linewidth=2, markersize=6)
    ax.axhline(80, color='gray', linestyle='--', alpha=0.5, label='A100 80GB')
    ax.set_xlabel('Model Size (B params)'); ax.set_ylabel('Memory per GPU (GB)')
    ax.set_title('ZeRO Memory Scaling (8 GPUs)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    # Plot 2: Pipeline bubble
    ax = axes[1]
    p_values = [2, 4, 8, 16]
    m_range = np.arange(1, 65)
    for p, color in zip(p_values, ['#E91E63', '#2196F3', '#4CAF50', '#FF9800']):
        bubbles = [bubble_fraction(p, m) * 100 for m in m_range]
        ax.plot(m_range, bubbles, label=f'p={p}', linewidth=2, color=color)
    ax.axhline(20, color='gray', linestyle=':', alpha=0.5)
    ax.text(60, 21, '20% target', fontsize=8, color='gray')
    ax.set_xlabel('Microbatches (m)'); ax.set_ylabel('Bubble Fraction (%)')
    ax.set_title('Pipeline Parallelism Efficiency')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('parallelism_memory.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: parallelism_memory.png")

## §5 Mixture-of-Experts (MoE) Training

MoE scales parameters without proportionally scaling FLOPs. Each token activates only top-$k$ of $E$ experts.

**Router (Gate) Function:**
$$G(\mathbf{x}) = \text{TopK}\left(\text{softmax}(\mathbf{W}_g \mathbf{x})\right)$$

**Load Balancing Loss** (prevents expert collapse):
$$\mathcal{L}_{\text{aux}} = \alpha \cdot E \cdot \sum_{i=1}^{E} f_i \cdot p_i$$

where $f_i$ = fraction of tokens routed to expert $i$, $p_i$ = mean router probability for expert $i$, $\alpha \approx 0.01$.

**Expert Capacity:**
$$C = k \cdot \frac{T}{E} \cdot c_f$$
where $T$ = tokens in batch, $c_f$ = capacity factor (typically 1.0–1.5).

In [ ]:
# === §5: Mixture-of-Experts Training ===

def softmax(x, temperature=1.0):
    """Numerically stable softmax."""
    x = x / temperature
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def moe_route(tokens, W_gate, top_k=2, noise_std=0.0):
    """
    Route tokens to experts.
    tokens: (T, d) — token embeddings
    W_gate: (d, E) — gate weights
    Returns: assignments, router_probs
    """
    logits = tokens @ W_gate
    if noise_std > 0:
        logits += np.random.randn(*logits.shape) * noise_std
    probs = softmax(logits)
    # Top-k selection
    top_k_idx = np.argsort(probs, axis=-1)[:, -top_k:]
    return top_k_idx, probs

def load_balancing_loss(assignments, probs, E, alpha=0.01):
    """
    Compute auxiliary load balancing loss.
    f_i = fraction of tokens sent to expert i
    p_i = mean probability assigned to expert i
    L_aux = alpha * E * sum(f_i * p_i)
    """
    T = len(assignments)
    # f_i: fraction of tokens routed to each expert
    f = np.zeros(E)
    for row in assignments:
        for idx in row:
            f[idx] += 1
    f /= (T * assignments.shape[1])  # normalise by total assignments
    
    # p_i: mean router probability for each expert
    p = probs.mean(axis=0)
    
    loss = alpha * E * np.sum(f * p)
    return loss, f, p

# --- Simulate MoE routing ---
np.random.seed(42)
T, d, E = 256, 64, 8  # 256 tokens, 64-dim, 8 experts
tokens = np.random.randn(T, d)

print("=== MoE Routing Simulation ===\n")
print(f"Tokens: {T}, Dimension: {d}, Experts: {E}, Top-k: 2\n")

# Case 1: Random (balanced) gate
W_balanced = np.random.randn(d, E) * 0.02
assign_bal, probs_bal = moe_route(tokens, W_balanced, top_k=2)
loss_bal, f_bal, p_bal = load_balancing_loss(assign_bal, probs_bal, E)

# Case 2: Biased gate (expert collapse)
W_biased = np.random.randn(d, E) * 0.02
W_biased[:, 0] += 0.5  # bias toward expert 0
W_biased[:, 1] += 0.3  # bias toward expert 1
assign_bias, probs_bias = moe_route(tokens, W_biased, top_k=2)
loss_bias, f_bias, p_bias = load_balancing_loss(assign_bias, probs_bias, E)

print("--- Expert Load Distribution ---\n")
print(f"{'Expert':<10} {'Balanced f_i':<16} {'Balanced p_i':<16} {'Biased f_i':<16} {'Biased p_i':<16}")
print("-" * 74)
for i in range(E):
    print(f"{i:<10} {f_bal[i]:<16.4f} {p_bal[i]:<16.4f} {f_bias[i]:<16.4f} {p_bias[i]:<16.4f}")
print(f"\n{'Ideal':<10} {1/E:<16.4f} {1/E:<16.4f}")

print(f"\nLoad balancing loss (balanced gate): {loss_bal:.6f}")
print(f"Load balancing loss (biased gate):   {loss_bias:.6f}")
print(f"Ratio: {loss_bias/loss_bal:.2f}× (higher = more imbalanced)")

# --- Expert capacity analysis ---
print("\n\n=== Expert Capacity Analysis ===\n")
print("Capacity C = k × (T/E) × c_f\n")
print(f"{'Cap Factor':<14} {'Capacity':<12} {'Overflow %':<14} {'Dropped Tokens'}")
print("-" * 54)
for cf in [1.0, 1.25, 1.5, 2.0]:
    capacity = int(2 * (T / E) * cf)  # top_k=2
    # Count tokens per expert
    counts = np.zeros(E, dtype=int)
    for row in assign_bias:
        for idx in row:
            counts[idx] += 1
    overflow = max(0, counts.max() - capacity)
    overflow_pct = overflow / T * 100
    print(f"{cf:<14} {capacity:<12} {overflow_pct:<14.1f} {overflow}")

# --- Simulate training with load balancing ---
print("\n\n=== Training MoE Gate with Load Balancing ===\n")
np.random.seed(42)
W_gate = np.random.randn(d, E) * 0.02
W_gate[:, 0] += 0.5  # start biased

steps = 200
lr_gate = 0.01
alpha = 0.01
losses = []
entropies = []

for step in range(steps):
    tokens_batch = np.random.randn(T, d)
    logits = tokens_batch @ W_gate
    probs_step = softmax(logits)
    top_k_idx = np.argsort(probs_step, axis=-1)[:, -2:]
    
    loss, f_step, p_step = load_balancing_loss(top_k_idx, probs_step, E, alpha)
    losses.append(loss)
    entropy = -np.sum(f_step * np.log(f_step + 1e-10))
    entropies.append(entropy)
    
    # Gradient: push toward uniform via load balancing
    # Simplified update: reduce weight for overloaded experts
    for i in range(E):
        excess = f_step[i] - 1/E
        W_gate[:, i] -= lr_gate * excess * np.mean(tokens_batch, axis=0)

if steps > 0:
    print(f"Initial entropy: {entropies[0]:.4f} (max = ln({E}) = {np.log(E):.4f})")
    print(f"Final entropy:   {entropies[-1]:.4f}")
    print(f"Initial loss:    {losses[0]:.6f}")
    print(f"Final loss:      {losses[-1]:.6f}")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Plot 1: Expert load — balanced vs biased
    ax = axes[0]
    x = np.arange(E)
    w = 0.35
    ax.bar(x - w/2, f_bal, w, label='Balanced', color='#2196F3', alpha=0.8)
    ax.bar(x + w/2, f_bias, w, label='Biased', color='#FF5722', alpha=0.8)
    ax.axhline(1/E, color='green', linestyle='--', label='Ideal (1/E)')
    ax.set_xlabel('Expert'); ax.set_ylabel('Token Fraction')
    ax.set_title('Expert Load Distribution')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Load balancing loss over training
    ax = axes[1]
    ax.plot(losses, linewidth=2, color='#2196F3')
    ax.set_xlabel('Step'); ax.set_ylabel('Aux Loss')
    ax.set_title('Load Balancing Loss'); ax.grid(True, alpha=0.3)
    
    # Plot 3: Entropy over training
    ax = axes[2]
    ax.plot(entropies, linewidth=2, color='#4CAF50')
    ax.axhline(np.log(E), color='red', linestyle='--', label=f'Max entropy (ln {E})')
    ax.set_xlabel('Step'); ax.set_ylabel('Routing Entropy')
    ax.set_title('Expert Utilisation Entropy')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('moe_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: moe_training.png")

## §6 Model FLOPs Utilisation (MFU)

MFU measures how efficiently hardware is used:

$$\text{MFU} = \frac{\text{Achieved FLOP/s}}{\text{Peak FLOP/s}} = \frac{6NT / t_{\text{step}}}{G \cdot F_{\text{peak}}}$$

where:
- $6N$ = FLOPs per token (forward + backward ≈ 6× parameters)
- $T$ = tokens per step
- $t_{\text{step}}$ = wall-clock time per step
- $G$ = number of GPUs
- $F_{\text{peak}}$ = peak FLOP/s per GPU

Good MFU: 40-60%. World-class: >55%. Theoretical max is ~100% but never achieved due to communication, memory, and non-matmul compute.

In [ ]:
# === §6: Model FLOPs Utilisation (MFU) ===

def compute_mfu(N_params, tokens_per_step, step_time_s, n_gpus, peak_flops_per_gpu):
    """
    Compute Model FLOPs Utilisation.
    N_params: total model parameters
    tokens_per_step: tokens processed per training step  
    step_time_s: wall-clock seconds per step
    n_gpus: number of GPUs
    peak_flops_per_gpu: theoretical peak FLOP/s per GPU
    """
    flops_per_step = 6 * N_params * tokens_per_step  # 6N per token (fwd + bwd)
    achieved_flops = flops_per_step / step_time_s
    peak_flops = n_gpus * peak_flops_per_gpu
    mfu = achieved_flops / peak_flops
    return mfu, achieved_flops, peak_flops

# --- Hardware specs ---
hardware = {
    'A100 (bf16)':  312e12,   # 312 TFLOP/s
    'H100 (bf16)':  989e12,   # 989 TFLOP/s
    'H100 (fp8)':   1979e12,  # 1979 TFLOP/s
    'B200 (bf16)':  2250e12,  # 2250 TFLOP/s (estimated)
}

# --- Worked example ---
print("=== Worked Example: MFU Calculation ===\n")
N = 7e9        # 7B parameters
T = 2e6        # 2M tokens per step
t = 1.2        # 1.2 seconds per step
G = 1024       # 1024 A100s
F = 312e12     # A100 bf16 peak

mfu, achieved, peak = compute_mfu(N, T, t, G, F)
print(f"Model:     7B parameters")
print(f"Tokens:    {T/1e6:.0f}M per step")
print(f"Step time: {t}s")
print(f"GPUs:      {G} × A100")
print(f"\nFLOPs/step  = 6 × {N:.0e} × {T:.0e} = {6*N*T:.2e}")
print(f"Achieved    = {6*N*T:.2e} / {t} = {achieved:.2e} FLOP/s")
print(f"Peak        = {G} × {F:.0e} = {peak:.2e} FLOP/s")
print(f"MFU         = {achieved:.2e} / {peak:.2e} = {mfu:.1%}")

# --- MFU for real training runs ---
print("\n\n=== MFU for Major Training Runs ===\n")
training_runs = [
    # (name, N_params, tokens/step, step_time, n_gpus, hw_key, reported_mfu)
    ("GPT-3",       175e9,  3.2e6,  60.0,  1024, 'A100 (bf16)', "~0.46"),
    ("Chinchilla",  70e9,   2.0e6,  20.0,  1024, 'A100 (bf16)', "~0.55"),
    ("PaLM",        540e9,  4.0e6,  120.0, 6144, 'A100 (bf16)', "~0.46"),
    ("LLaMA-65B",   65e9,   4.0e6,  55.0,  2048, 'A100 (bf16)', "~0.43"),
    ("LLaMA-3 70B", 70e9,   4.0e6,  8.0,   2048, 'H100 (bf16)', "~0.38"),
]

print(f"{'Model':<16} {'N':<10} {'T/step':<10} {'t(s)':<8} {'GPUs':<8} {'MFU (calc)':<12} {'Reported'}")
print("-" * 78)
for name, N, T, t, G, hw, reported in training_runs:
    mfu_val, _, _ = compute_mfu(N, T, t, G, hardware[hw])
    print(f"{name:<16} {N/1e9:.0f}B{'':<5} {T/1e6:.1f}M{'':<4} {t:<8.1f} {G:<8} {mfu_val:<12.1%} {reported}")

# --- FLOPs breakdown by operation ---
print("\n\n=== FLOPs Breakdown by Operation (per layer, per token) ===\n")
def flops_breakdown(d, d_ff, s, n_heads):
    """FLOPs per token per layer for a Transformer."""
    d_k = d // n_heads
    ops = {
        'QKV projection': 3 * 2 * d * d,              # 3 projections
        'Attention scores': 2 * s * d,                  # QK^T
        'Attention × V': 2 * s * d,                     # softmax(QK^T)V
        'Output projection': 2 * d * d,                 # W_O
        'FFN up': 2 * d * d_ff,                         # W1
        'FFN down': 2 * d_ff * d,                       # W2
    }
    return ops

d, d_ff, s, n_heads = 4096, 11008, 2048, 32  # LLaMA-7B config
ops = flops_breakdown(d, d_ff, s, n_heads)
total = sum(ops.values())

print(f"LLaMA-7B config: d={d}, d_ff={d_ff}, seq={s}, heads={n_heads}\n")
print(f"{'Operation':<22} {'FLOPs/token':<16} {'% Total'}")
print("-" * 48)
for name, flops in ops.items():
    print(f"{name:<22} {flops:<16,} {flops/total*100:.1f}%")
print(f"{'Total':<22} {total:<16,} 100.0%")
print(f"\n6N approximation: {6*7e9:.2e} vs actual per-layer×32: {total*32:.2e}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    names = list(ops.keys())
    values = [v/1e6 for v in ops.values()]
    colors = ['#2196F3', '#FF9800', '#FF5722', '#4CAF50', '#9C27B0', '#00BCD4']
    ax.barh(names, values, color=colors, alpha=0.8)
    ax.set_xlabel('MFLOPs per token per layer')
    ax.set_title('FLOPs Breakdown (LLaMA-7B)')
    ax.grid(True, alpha=0.3, axis='x')
    for i, v in enumerate(values):
        ax.text(v + 0.5, i, f'{v:.0f}M', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('flops_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: flops_breakdown.png")

## §7 LoRA: Low-Rank Adaptation

Instead of fine-tuning all $N$ parameters, LoRA freezes the base model and adds low-rank updates:

$$W' = W + \Delta W = W + BA$$
where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, and $r \ll d$.

**Trainable parameters per adapted weight:**
$$P_{\text{LoRA}} = d \times r + r \times d = 2dr$$

**vs full fine-tuning:** $d^2$ → reduction factor: $\frac{d}{2r}$

Typical: $r \in \{4, 8, 16, 32, 64\}$, apply to Q, V (or Q, K, V, O + FFN for QLoRA/DoRA).

In [ ]:
# === §7: LoRA Parameter Counting ===

def lora_params(d, r, target_modules, n_layers):
    """
    Count LoRA trainable parameters.
    d: model hidden dimension
    r: LoRA rank
    target_modules: list of module names (each assumed d×d)
    n_layers: number of transformer layers
    """
    params_per_module = 2 * d * r  # B(d×r) + A(r×d)
    total = params_per_module * len(target_modules) * n_layers
    return total

def full_finetune_params(d, d_ff, n_layers, vocab_size):
    """Total parameters for a Transformer."""
    attn = 4 * d * d  # Q, K, V, O
    ffn = 2 * d * d_ff + d_ff  # up + down + bias (SwiGLU has 3×)
    per_layer = attn + ffn + 2 * d  # + layer norms
    total = per_layer * n_layers + vocab_size * d  # + embedding
    return total

# --- Worked example: LLaMA-7B ---
print("=== LoRA Parameter Count: LLaMA-7B ===\n")
d, d_ff, n_layers, vocab = 4096, 11008, 32, 32000
full_params = full_finetune_params(d, d_ff, n_layers, vocab)

configs = [
    ("Q, V only", ["Q", "V"], [4, 8, 16, 32, 64]),
    ("Q, K, V, O", ["Q", "K", "V", "O"], [4, 8, 16, 32]),
    ("All linear", ["Q", "K", "V", "O", "up", "down"], [4, 8, 16]),
]

for name, modules, ranks in configs:
    print(f"\n--- Target: {name} ({len(modules)} modules × {n_layers} layers) ---\n")
    print(f"{'Rank r':<10} {'LoRA Params':<18} {'% of Full':<12} {'Reduction'}")
    print("-" * 52)
    for r in ranks:
        lora_p = lora_params(d, r, modules, n_layers)
        pct = lora_p / full_params * 100
        reduction = full_params / lora_p
        print(f"{r:<10} {lora_p:>14,}   {pct:<12.3f} {reduction:,.0f}×")

# --- LoRA forward pass simulation ---
print("\n\n=== LoRA Forward Pass Simulation ===\n")
np.random.seed(42)
d_sim, r_sim = 64, 4  # small for demo

# Original weight
W = np.random.randn(d_sim, d_sim) * 0.02
# LoRA matrices (A initialised with Kaiming, B with zeros)
A = np.random.randn(r_sim, d_sim) * np.sqrt(2.0 / d_sim)  # Kaiming
B = np.zeros((d_sim, r_sim))  # zero init → ΔW starts as 0

x = np.random.randn(d_sim)
# Before training: LoRA output = 0
delta_W = B @ A
y_original = W @ x
y_lora = (W + delta_W) @ x

print(f"x norm:         {np.linalg.norm(x):.4f}")
print(f"‖W‖_F:          {np.linalg.norm(W):.4f}")
print(f"‖ΔW‖_F (init):  {np.linalg.norm(delta_W):.4f}  (B=0 → ΔW=0)")
print(f"y_original:     ‖y‖ = {np.linalg.norm(y_original):.4f}")
print(f"y_lora (init):  ‖y‖ = {np.linalg.norm(y_lora):.4f}  (identical → safe init)")

# After training: B gets updated
B_trained = np.random.randn(d_sim, r_sim) * 0.1
delta_W_trained = B_trained @ A
y_lora_trained = (W + delta_W_trained) @ x
print(f"\nAfter LoRA training:")
print(f"‖ΔW‖_F:         {np.linalg.norm(delta_W_trained):.4f}")
print(f"‖ΔW‖/‖W‖:       {np.linalg.norm(delta_W_trained)/np.linalg.norm(W):.4f}")
print(f"y_lora_trained:  ‖y‖ = {np.linalg.norm(y_lora_trained):.4f}")

# --- Rank Analysis: SVD of ΔW ---
print("\n\n=== LoRA Rank Analysis ===\n")
print("ΔW = BA has rank at most r. SVD shows the effective structure:\n")
U, S, Vt = np.linalg.svd(delta_W_trained)
print(f"{'Sing. Value':<16} {'Magnitude':<14} {'% Energy'}")
print("-" * 42)
total_energy = np.sum(S**2)
for i, s in enumerate(S[:10]):
    pct = s**2 / total_energy * 100 if total_energy > 0 else 0
    marker = " ← rank r" if i == r_sim - 1 else ""
    print(f"σ_{i+1}{'':<12} {s:<14.6f} {pct:.1f}%{marker}")
print(f"\nNon-zero singular values: {np.sum(S > 1e-10)} (should be ≤ r={r_sim})")

# --- Comparison table ---
print("\n\n=== Fine-Tuning Methods Comparison ===\n")
methods_data = [
    ["Full Fine-Tune", "100%", "16N bytes", "All"],
    ["LoRA (r=16, QV)", "0.1-0.3%", "~1GB", "B, A matrices"],
    ["QLoRA", "0.1-0.3%", "~0.5GB", "4-bit base + LoRA"],
    ["DoRA", "0.15-0.35%", "~1.2GB", "LoRA + direction"],
    ["Prefix Tuning", "0.05-0.1%", "~0.3GB", "Prefix embeddings"],
    ["Adapter", "0.5-2%", "~2GB", "Bottleneck layers"],
]
print_table(
    ["Method", "Trainable %", "Memory", "What's Trained"],
    methods_data
)

## §8 Numerical Stability & Initialisation

Deep networks amplify variance through layers. Proper initialisation and normalisation prevent explosion/vanishing.

**Variance propagation:** For layer $\mathbf{y} = W\mathbf{x}$, if $W_{ij} \sim \mathcal{N}(0, \sigma^2)$:
$$\text{Var}(y_j) = d_{\text{in}} \cdot \sigma^2 \cdot \text{Var}(x)$$

**Xavier:** $\sigma^2 = \frac{2}{d_{\text{in}} + d_{\text{out}}}$ — preserves variance for linear activations.

**He (Kaiming):** $\sigma^2 = \frac{2}{d_{\text{in}}}$ — accounts for ReLU killing half the activations.

**GPT-2 Residual Scaling:** Scale residual projection by $\frac{1}{\sqrt{2L}}$ where $L$ = number of layers.

**QK-Norm:** Pre-normalise queries and keys before attention to prevent logit growth:
$$\text{Attn} = \text{softmax}\left(\frac{\text{norm}(Q) \cdot \text{norm}(K)^T}{\sqrt{d_k}}\right)V$$

In [ ]:
# === §8: Numerical Stability & Initialisation ===

def variance_propagation(d_in, d_out, sigma, n_layers, activation='linear'):
    """Track variance through n_layers."""
    var = 1.0  # input variance
    history = [var]
    for _ in range(n_layers):
        var *= d_in * sigma**2
        if activation == 'relu':
            var *= 0.5  # ReLU kills half
        elif activation == 'gelu':
            var *= 0.425  # GELU approximation
        history.append(var)
    return history

# --- Compare initialisations ---
print("=== Variance Propagation Through 96 Layers (d=4096) ===\n")
d = 4096
n_layers = 96

inits = {
    'std=0.02 (GPT)': 0.02,
    'Xavier': np.sqrt(2.0 / (d + d)),
    'He (Kaiming)': np.sqrt(2.0 / d),
    'Too large (0.1)': 0.1,
    'Too small (0.001)': 0.001,
}

print(f"{'Init':<22} {'σ':<12} {'Var @ L=1':<14} {'Var @ L=32':<14} {'Var @ L=96':<14} {'Status'}")
print("-" * 90)
all_histories = {}
for name, sigma in inits.items():
    history = variance_propagation(d, d, sigma, n_layers, 'linear')
    all_histories[name] = history
    v1 = history[1]
    v32 = history[32] if len(history) > 32 else float('inf')
    v96 = history[96] if len(history) > 96 else float('inf')
    status = "✓ stable" if 0.1 < v96 < 10 else ("💥 explode" if v96 > 10 else "📉 vanish")
    print(f"{name:<22} {sigma:<12.6f} {v1:<14.4e} {v32:<14.4e} {v96:<14.4e} {status}")

# --- GPT-2 residual scaling ---
print("\n\n=== GPT-2 Residual Scaling: 1/√(2L) ===\n")
print("In a residual network: x_l = x_{l-1} + f(x_{l-1})")
print("After L layers: Var(x_L) ≈ Var(x_0) + L·Var(f)")
print("→ Scale residual by 1/√(2L) to keep Var(x_L) ≈ Var(x_0)\n")

for L in [12, 24, 32, 48, 96]:
    scale = 1.0 / np.sqrt(2 * L)
    print(f"L = {L:<4} → scale = {scale:.6f}")

# --- Simulate deep network with different inits ---
print("\n\n=== Deep Network Simulation (96 layers, d=256) ===\n")
np.random.seed(42)
d_sim = 256
n_sim = 96

def simulate_deep_network(d, n_layers, init_fn, use_residual_scale=False, use_layernorm=False):
    """Forward pass through n_layers linear layers."""
    x = np.random.randn(d) / np.sqrt(d)
    norms = [np.linalg.norm(x)]
    
    for l in range(n_layers):
        W = init_fn(d) 
        h = W @ x
        if use_residual_scale:
            h *= 1.0 / np.sqrt(2 * n_layers)
        if use_layernorm:
            h = (h - h.mean()) / (h.std() + 1e-8)
        x = x + h  # residual connection
        norms.append(np.linalg.norm(x))
    return norms

configs_sim = {
    'std=0.02': lambda d: np.random.randn(d, d) * 0.02,
    'Xavier': lambda d: np.random.randn(d, d) * np.sqrt(2.0 / (d + d)),
    'Xavier + res_scale': lambda d: np.random.randn(d, d) * np.sqrt(2.0 / (d + d)),
    'Xavier + LayerNorm': lambda d: np.random.randn(d, d) * np.sqrt(2.0 / (d + d)),
}

print(f"{'Config':<24} {'‖x‖ @ L=1':<14} {'‖x‖ @ L=32':<14} {'‖x‖ @ L=96':<14}")
print("-" * 66)
all_norms = {}
for name, init_fn in configs_sim.items():
    np.random.seed(42)
    res_scale = 'res_scale' in name
    ln = 'LayerNorm' in name
    norms = simulate_deep_network(d_sim, n_sim, init_fn, res_scale, ln)
    all_norms[name] = norms
    print(f"{name:<24} {norms[1]:<14.4f} {norms[32]:<14.4f} {norms[96]:<14.4f}")

# --- QK-Norm demonstration ---
print("\n\n=== QK-Norm: Preventing Attention Logit Growth ===\n")
np.random.seed(42)
d_k = 64
seq_len = 128

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)

# Simulate what happens as training proceeds (magnitudes grow)
print(f"{'Scale Factor':<14} {'Max logit':<14} {'logit std':<14} {'Max logit (QK-norm)':<22} {'logit std (QK-norm)'}")
print("-" * 80)
for scale in [1.0, 2.0, 5.0, 10.0, 50.0]:
    Qs = Q * scale
    Ks = K * scale
    
    # Without QK-norm
    logits = Qs @ Ks.T / np.sqrt(d_k)
    
    # With QK-norm
    Qn = Qs / (np.linalg.norm(Qs, axis=-1, keepdims=True) + 1e-8)
    Kn = Ks / (np.linalg.norm(Ks, axis=-1, keepdims=True) + 1e-8)
    logits_norm = Qn @ Kn.T / np.sqrt(d_k)
    
    print(f"{scale:<14.1f} {np.max(np.abs(logits)):<14.2f} {logits.std():<14.4f} "
          f"{np.max(np.abs(logits_norm)):<22.4f} {logits_norm.std():<.4f}")

print("\n→ Without QK-norm: logits grow with scale → softmax saturates → gradient vanishes")
print("→ With QK-norm: logits stay bounded regardless of Q,K magnitude growth")

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Activation norms through layers
    ax = axes[0]
    colors = ['#E91E63', '#2196F3', '#4CAF50', '#FF9800']
    for (name, norms), color in zip(all_norms.items(), colors):
        ax.plot(norms, label=name, linewidth=2, color=color)
    ax.set_xlabel('Layer'); ax.set_ylabel('‖x‖₂')
    ax.set_title('Activation Norm Through 96 Layers')
    ax.set_yscale('symlog'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    # Plot 2: Variance propagation
    ax = axes[1]
    for (name, history), color in zip(list(all_histories.items())[:4], colors):
        ax.plot(history[:50], label=name, linewidth=2, color=color)
    ax.set_xlabel('Layer'); ax.set_ylabel('Variance')
    ax.set_title('Variance Propagation (first 50 layers)')
    ax.set_yscale('symlog'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('numerical_stability.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: numerical_stability.png")

## §9 Full Training Loop Simulation

Putting it all together: initialisation → forward → loss → backward → gradient clipping → Adam step → LR schedule.

We simulate training a small 2-layer Transformer on a toy language modelling task, demonstrating every mathematical concept from this chapter in a single coherent training loop.

In [ ]:
# === §9: Full Training Loop Simulation ===
# Ties together: Adam, gradient clipping, LR schedule, numerical stability

np.random.seed(42)

# --- Toy problem: minimise Rosenbrock function ---
# f(x,y) = (1-x)² + 100(y-x²)²
# This is notoriously hard — narrow curved valley, tests optimiser robustness

def rosenbrock(theta):
    x, y = theta
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(theta):
    x, y = theta
    dx = -2*(1 - x) + 200*(y - x**2)*(-2*x)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

print("=== Full Training Loop: Rosenbrock Optimisation ===\n")
print("f(x,y) = (1-x)² + 100(y-x²)²")
print("Minimum at (1, 1), f(1,1) = 0")
print("Starting from (-1, 1), f(-1,1) = 4\n")

# Configuration
total_steps = 2000
warmup_steps = 200
clip_threshold = 10.0
lr_max = 0.002
lr_min = 1e-5

# Initialise
theta = np.array([-1.0, 1.0])
opt = Adam([theta], lr=lr_max, beta1=0.9, beta2=0.999)

# Training log
log = {'step': [], 'loss': [], 'lr': [], 'grad_norm': [], 'clipped': [],
       'theta_x': [], 'theta_y': []}

for step in range(total_steps):
    # 1. Compute loss and gradient
    loss = rosenbrock(theta)
    grad = rosenbrock_grad(theta)
    
    # Add noise (SGD simulation)
    grad_noisy = grad + np.random.randn(2) * 0.5
    
    # 2. Gradient clipping
    grad_clipped, g_norm, was_clipped = clip_by_norm(grad_noisy, clip_threshold)
    
    # 3. LR schedule (warmup + cosine)
    lr = warmup_cosine(step, lr_max, lr_min, warmup_steps, total_steps)
    opt.lr = lr
    
    # 4. Adam step
    opt.step([grad_clipped])
    
    # Log
    log['step'].append(step)
    log['loss'].append(loss)
    log['lr'].append(lr)
    log['grad_norm'].append(g_norm)
    log['clipped'].append(was_clipped)
    log['theta_x'].append(theta[0])
    log['theta_y'].append(theta[1])

# --- Print training summary ---
print(f"{'Step':<8} {'Loss':<14} {'LR':<14} {'‖g‖':<12} {'Clipped':<10} {'θ'}")
print("-" * 78)
for i in [0, 50, 100, 200, 500, 1000, 1500, 1999]:
    print(f"{log['step'][i]:<8} {log['loss'][i]:<14.6f} {log['lr'][i]:<14.6e} "
          f"{log['grad_norm'][i]:<12.4f} {'Yes' if log['clipped'][i] else 'No':<10} "
          f"({log['theta_x'][i]:.4f}, {log['theta_y'][i]:.4f})")

print(f"\nFinal position: ({theta[0]:.6f}, {theta[1]:.6f})")
print(f"Final loss: {rosenbrock(theta):.8f}")
print(f"Distance to optimum: {np.linalg.norm(theta - np.array([1.0, 1.0])):.6f}")
print(f"Clipping events: {sum(log['clipped'])} / {total_steps} ({sum(log['clipped'])/total_steps*100:.1f}%)")

# --- Compare optimisers on same problem ---
print("\n\n=== Optimiser Comparison on Rosenbrock ===\n")

def train_rosenbrock(opt_class_args, steps=2000, clip=10.0, seed=42):
    np.random.seed(seed)
    theta = np.array([-1.0, 1.0])
    opt = Adam([theta], **opt_class_args)
    losses = []
    for step in range(steps):
        loss = rosenbrock(theta)
        grad = rosenbrock_grad(theta) + np.random.randn(2) * 0.5
        grad, _, _ = clip_by_norm(grad, clip)
        lr = warmup_cosine(step, opt_class_args.get('lr', 1e-3), 1e-5, 200, steps)
        opt.lr = lr
        opt.step([grad])
        losses.append(loss)
    return losses, theta

configs = {
    'Adam (β₁=0.9)': {'lr': 2e-3, 'beta1': 0.9, 'beta2': 0.999},
    'Adam (β₁=0.95)': {'lr': 2e-3, 'beta1': 0.95, 'beta2': 0.999},
    'AdamW (wd=0.01)': {'lr': 2e-3, 'beta1': 0.9, 'beta2': 0.999, 'weight_decay': 0.01},
    'Adam (β₂=0.95)': {'lr': 2e-3, 'beta1': 0.9, 'beta2': 0.95},
}

print(f"{'Config':<22} {'Final Loss':<14} {'Final θ':<24} {'Dist to opt'}")
print("-" * 74)
all_opt_losses = {}
for name, args in configs.items():
    losses, final_theta = train_rosenbrock(args)
    all_opt_losses[name] = losses
    dist = np.linalg.norm(final_theta - np.array([1.0, 1.0]))
    print(f"{name:<22} {losses[-1]:<14.6f} ({final_theta[0]:.4f}, {final_theta[1]:.4f}){'':<6} {dist:.6f}")

if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Loss curve
    ax = axes[0, 0]
    ax.plot(log['loss'], linewidth=1.5, color='#2196F3')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Training Loss (Rosenbrock)'); ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    
    # Plot 2: LR schedule + gradient norm
    ax = axes[0, 1]
    ax2 = ax.twinx()
    ax.plot(log['lr'], 'b-', linewidth=1.5, label='Learning Rate')
    ax2.plot(log['grad_norm'], 'r-', linewidth=0.5, alpha=0.5, label='Gradient Norm')
    ax2.axhline(clip_threshold, color='red', linestyle='--', alpha=0.3)
    ax.set_xlabel('Step'); ax.set_ylabel('LR', color='blue')
    ax2.set_ylabel('‖g‖', color='red')
    ax.set_title('LR Schedule & Gradient Norm')
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Trajectory on Rosenbrock
    ax = axes[1, 0]
    x_range = np.linspace(-1.5, 1.5, 200)
    y_range = np.linspace(-0.5, 2.0, 200)
    X, Y = np.meshgrid(x_range, y_range)
    Z = (1 - X)**2 + 100*(Y - X**2)**2
    ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), alpha=0.4, cmap='viridis')
    ax.plot(log['theta_x'][::10], log['theta_y'][::10], 'r-', linewidth=0.8, alpha=0.7)
    ax.plot(log['theta_x'][0], log['theta_y'][0], 'go', markersize=10, label='Start')
    ax.plot(1, 1, 'r*', markersize=15, label='Optimum')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title('Adam Trajectory on Rosenbrock')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    
    # Plot 4: Optimiser comparison
    ax = axes[1, 1]
    colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
    for (name, losses), color in zip(all_opt_losses.items(), colors):
        ax.plot(losses[::10], label=name, linewidth=1.5, color=color)
    ax.set_xlabel('Step (÷10)'); ax.set_ylabel('Loss')
    ax.set_title('Optimiser Comparison'); ax.set_yscale('log')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_loop.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: training_loop.png")

## §10 Summary

| Topic | Key Equation | Why It Matters |
|-------|-------------|----------------|
| **Adam** | $\theta \leftarrow \theta - \eta \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$ | Adaptive LR per parameter, de facto standard |
| **AdamW** | Decoupled $\theta \leftarrow \theta - \eta\lambda\theta$ | Clean regularisation, better generalisation |
| **Gradient Clipping** | $\min(1, \tau/\|\mathbf{g}\|_2) \cdot \mathbf{g}$ | Prevents training divergence from loss spikes |
| **Cosine LR** | $\eta_{\min} + \frac{1}{2}(\eta_{\max}-\eta_{\min})(1+\cos(\pi t/T))$ | Smooth annealing, better final loss |
| **WSD Schedule** | Warmup → Stable → Decay | Enables continual training and branching |
| **ZeRO-3** | $16N/K$ bytes per GPU | Makes 100B+ models trainable |
| **Pipeline Bubble** | $(p-1)/(p-1+m)$ | Quantifies parallelism overhead |
| **MoE Aux Loss** | $\alpha E \sum f_i p_i$ | Prevents expert collapse |
| **MFU** | $6NT / (t \cdot G \cdot F_{\text{peak}})$ | Measures hardware utilisation |
| **LoRA** | $W' = W + BA$, params = $2dr$ | 100-1000× fewer trainable params |
| **Residual Scale** | $1/\sqrt{2L}$ | Prevents variance explosion in deep nets |
| **QK-Norm** | Normalise Q, K before attention | Bounds logit growth, stabilises training |

---
*Next: [Fine-Tuning Math →](../07-Fine-Tuning-Math/notes.md)*